In [1]:
import numpy as np
import matplotlib.pyplot as plt 
import os 
import sys 
from PIL import Image, ImageOps 
import math

Read in the file 

In [2]:
def retrieveFile(file): 
    filepath = os.path.join(os.getcwd(), file)   
    image = Image.open(filepath)  
    image = ImageOps.grayscale(image)
    dim = max(image.size) 
    new_dim = 2 ** int(math.ceil(math.log(dim, 2)))
    return ImageOps.pad(image, (new_dim, new_dim))

Gets the har step which is used to average the output of the two input columns

In [3]:
def getHarStep(i, k):  
    transform = np.zeros((2 ** k, 2 ** k))  

    for j in range(2 ** (k - i - 1)): 
        transform[2 * j, j] = 0.5 
        transform[2 * j + 1, j] = 0.5 

    offset = 2 ** (k - i - 1) 
    for j in range(offset): 
        transform[2 * j, offset + j] = 0.5 
        transform[2 * j + 1, offset + j] = -0.5 
        
    for j in range(2 ** (k - i), 2 ** k): 
        transform[j, j] = 1 
    return transform



itterate through all the columns in the image

In [4]:
def getHarTransform(k): 
    transform = np.eye(2 ** k) 
    for i in range(k): 
        transform =  transform @ getHarStep(i, k)
    return transform

encodes the image using the har transform

In [5]:
def haarEncode(array: np.ndarray): 
    k = int(np.ceil(np.log2(len(array))))  
    print(f"Encoding array of shape {array.shape} with k={k}")
    # assert array.shape == (k,k) 
    rowEncoder = getHarTransform(k) 
    return rowEncoder.T @ array @ rowEncoder

Decodes the har transformed image using by inverting the input transformatioon

In [6]:
def haarDecode(array: np.ndarray): 
    k = int(np.ceil(np.log2(len(array)))) 
    print(f"Decoding array of shape {array.shape} with k={k}")
    rowDecoder = np.linalg.inv(getHarTransform(k))
    return rowDecoder @ array @ rowDecoder.T

Truncating small non zero values for better compression

In [7]:
def truncateValues(array, threshold): 
    return np.where(np.abs(array) < threshold, 0, array)

In [8]:
threshold = 8 

print("Input target image") 
input_image = input()  

Input target image


In [9]:

image = retrieveFile(input_image)  
array = np.array(image) 
encode = haarEncode(array)   
encode = truncateValues(encode, threshold) 
decode = haarDecode(encode)  
compressedImage = Image.fromarray(np.uint8(decode)) 
compressedImage.show()

Encoding array of shape (16384, 16384) with k=14


KeyboardInterrupt: 